In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import os
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
from snowflake.snowpark.functions import col, dateadd, current_date, date_trunc, lit

In [2]:
connection_parameters = {
    "account": "yva20138.us-west-2.privatelink",
    "user": "ben.pharris@dish.com",
    "authenticator": "externalbrowser",
    "insecure_mode":True,
}

session = Session.builder.configs(connection_parameters).create()

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://dish.okta.com/app/snowflake/exk9ewak82R5btDoW2p7/sso/saml?SAMLRequest=pZJNc9owEIb%2Fikc925JFKKDBZNzQtLRpy%2FCRTHIT9gIabMnRyhj66ytM6KSH5NKbR35W%2B2jfHV4fyiLYg0VldELiiJEAdGZypTcJWS5uwz4J0Emdy8JoSMgRkFyPhijLohJp7bZ6Bs81oAv8RRpF%2ByMhtdXCSFQotCwBhcvEPP1xJ3jERGWNM5kpyKuS9yskIljnDS8lOSqvt3WuEpQ2TRM1ncjYDeWMMcoG1FMn5MOFP%2Fg3vcHHlF2deE94fPri9knp8wje01qdIRRfF4tpOP01X5AgvajeGI11CXYOdq8yWM7uzgLoDR7vU87iTj%2BqMWz87EIeVVbtpYNC6V2E2jTrQu4gM2VVO98i8l90DTktzEb5KUzGCal2KmcPm87qaB8d8rg5bFD1vjyt6qr3Pf3GBsAPGQxKjb8Hn5%2BXGQnuLzHzU8wTxBom%2BhSu80eMd8M4DmO%2B4Ex0O%2BKqF3U%2Fdp9IMPaCSkvXVl5ekCvcRmbnZGsmq4r%2BlaZw2A2gkbs%2Bn3VXbmweeNWjiIaegibn3RFtdzv6r4kM6eurXnbyp49pMp6aQmXH4NbYUrq3U4yjuD1RebhuUQGlVEWa5xYQfZpFYZobC94jIc7WQOjo3PXf5R%2F9AQ%3D%3D&RelayState=v

In [ ]:
# Sales Channel GAs

query = """
SELECT
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD,
    SUM(GROSS_ACTIVATION_QUANTITY) AS TOTAL_GROSS_ACTIVATION
FROM
    SBX.SBX_WRLSPPDSLSOPS.RADAR_GAS
WHERE
    ACTIVATION_DATE BETWEEN DATE_TRUNC('QUARTER', CURRENT_DATE) AND CURRENT_DATE
GROUP BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
ORDER BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
"""
ga_snowflake = session.sql(query).to_pandas()


In [ ]:

slsops_to_sales_channel = {
    'Direct Distribution Partner': 'Indirect',
    'Axiom Indirect': 'Digital',
    'Dish Internal': 'Digital',
    'Web Sales': 'Digital',
    'Amazon': 'Amazon',
    'Best Buy': 'National Retail',
    'National Retailer': 'National Retail',
    'WalMart': 'National Retail',
    'Target': 'National Retail',
    'The Kroger Co.': 'National Retail',
    'Cross Sell': 'Cross Sell',
    'Direct': 'Telesales',
    'Apple': 'Apple'
}

ga = ga_snowflake.dropna() 

ga['Channel'] = ga_snowflake['SLSOPS_CHANNEL_RECORD'].map(slsops_to_sales_channel)

ga.drop(['SLSOPS_CHANNEL_RECORD'], axis=1)


In [3]:
spends = session.table('SBX.SBX_WRLSPPDSLSOPS.AH0027_AN_TEMP_MARKETING_PACING_DATAMART').to_pandas()

In [ ]:

spends['DATE'] = pd.to_datetime(spends['DATE'])

daily_sales_channel = spends.groupby(['FUNNEL', 'CHANNEL', 'DATE'], as_index=False).sum().sort_values(['DATE','CHANNEL'])

assert daily_sales_channel[daily_sales_channel['CHANNEL'] != 'All']['SPEND'].sum() == daily_sales_channel[daily_sales_channel['CHANNEL'] == 'All']['SPEND'].sum()


AssertionError: 

In [5]:
daily_sales_channel['SPEND'] = pd.to_numeric(daily_sales_channel['SPEND'], errors='coerce')
daily_sales_channel['SPEND'] = daily_sales_channel['SPEND'].map('{:.2f}'.format)
daily_sales_channel.to_csv("/Users/Ben.Pharris/Documents/project-dev/Ad Hoc/Atlas Reporting/spends.csv")

In [ ]:
query = 'SELECT * FROM SBX.SBX_WRLSPPDSLSOPS.RADAR_GAS limit 100'

conv_ex = session.sql(query).to_pandas()

conv_ex.to_csv("/Users/Ben.Pharris/Documents/project-dev/Ad Hoc/Atlas Reporting/conv_ex.csv")

In [ ]:
t = session.table("DISH_RETAIL_DL.ORDER_ORCHESTRATION.CUSTOMERORDER_PARSE")

In [ ]:
cutoff = dateadd("quarter", lit(-5), date_trunc("quarter", current_date()))

df_recent = t.filter(col("OO_LASTMODIFIED_DT_MT") >= cutoff)

df_recent.count()

In [ ]:
import pandas as pd

xGtW>)f%>YnF3S#H


spend = pd.read_parquet("/Users/Ben.Pharris/Documents/project-dev/Media Measurement/data/clean/spend_cleaned.parquet")